# Train DistilBERT on Colab (free GPU)

This notebook fine-tunes DistilBERT for the AI Health Assistant using the code in `src/bert_model.py`.
It uses exactly the same train / validation / test splits and labels as the baseline.

**Steps**
1. Check the GPU
2. Get the code from GitHub
3. Install the requirements
4. Quick smoke test (about 1 minute)
5. Train with learning rate 5e-5, then 3e-5 - validation only, test set untouched
6. Pick the winner by **validation** macro-F1
7. Evaluate the winner **once** on the test set
8. Zip `models/bert` and `reports/` and download them

> Educational project only - not a medical diagnosis tool.

## Step 1 - Check the GPU

In the menu choose **Runtime -> Change runtime type -> T4 GPU**, then run the cell below.
You should see a Tesla T4 and `CUDA available: True`. If it says `False`, training still works but will be much slower.

In [1]:
!nvidia-smi
import torch
print("CUDA available:", torch.cuda.is_available())

Fri Nov  6 18:06:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   57C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Step 2 - Get the code

Clone the project repository and move into it. All later commands run from the project folder.

In [2]:
!git clone https://github.com/zaima-20/ai-health-assistant.git
%cd ai-health-assistant
!ls

Cloning into 'ai-health-assistant'...
remote: Enumerating objects: 38, done.
remote: Counting objects: 100% (38/38), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 38 (delta 1), reused 38 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (38/38), 381.38 KiB | 2.80 MiB/s, done.
Resolving deltas: 100% (1/1), done.
/content/ai-health-assistant
app.py	   data    notebooks   README.md  requirements.txt  tests
CLAUDE.md  models  pytest.ini  reports	  src


## Step 3 - Install the requirements

Colab already has PyTorch and most libraries, so this is usually quick.

In [3]:
!pip install -q -r requirements.txt

## Step 4 - Smoke test (optional, about 1 minute)

Trains for 1 epoch on 40 rows to check that everything runs. It does **not** use the test set
and does **not** write anything to `models/` or `reports/`.

In [4]:
!python -m src.bert_model --smoke

README.md: 100% 2.46k/2.46k [00:00<00:00, 7.86MB/s]
train.jsonl: 100% 171k/171k [00:00<00:00, 83.7MB/s]
test.jsonl: 100% 42.5k/42.5k [00:00<00:00, 81.7MB/s]
Generating train split: 100% 853/853 [00:00<00:00, 41306.26 examples/s]
Generating test split: 100% 212/212 [00:00<00:00, 109655.01 examples/s]
config.json: 100% 483/483 [00:00<00:00, 1.93MB/s]
tokenizer_config.json: 100% 48.0/48.0 [00:00<00:00, 243kB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 2.62MB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 3.59MB/s]

model.safetensors: downloading bytes:  89% 239M/268M [00:01<00:00, 251MB/s, 21.3MB/s  ]
model.safetensors: reconstructing file:  81% 218M/268M [00:01<00:00, 142MB/s]
model.safetensors: downloading bytes: 100% 250M/250M [00:01<00:00, 147MB/s, 23.2MB/s  ]
model.safetensors: reconstructing file: 100% 268M/268M [00:01<00:00, 157MB/s, 25.5MB/s  ]
Loading weights: 100% 100/100 [00:00<00:00, 17040.32it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-

## Step 5a - Train with learning rate 5e-5 (validation only)

Up to 8 epochs, with early stopping on validation macro-F1 (patience 2). The best epoch is kept.
`--skip-test` means the test set is **not touched** here.
The model and validation results are copied into `runs/lr_5e-5/` so the next run does not overwrite them.

In [5]:
!python -m src.bert_model --lr 5e-5 --skip-test
!rm -rf runs/lr_5e-5 && mkdir -p runs/lr_5e-5 && cp -r models/bert runs/lr_5e-5/ && cp reports/bert_metrics.json reports/bert_confusion_val.png runs/lr_5e-5/

Device: cuda  |  train=720 val=128  |  lr=5e-05  |  test set skipped
Loading weights: 100% 100/100 [00:00<00:00, 4228.47it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
epoch 1: train_loss=3.0351 val_acc=0.4062 val_macro_f1=0.3382  <- best
epoch 2: train_loss=2.

## Step 5b - Train with learning rate 3e-5 (validation only)

In [6]:
!python -m src.bert_model --lr 3e-5 --skip-test
!rm -rf runs/lr_3e-5 && mkdir -p runs/lr_3e-5 && cp -r models/bert runs/lr_3e-5/ && cp reports/bert_metrics.json reports/bert_confusion_val.png runs/lr_3e-5/

Device: cuda  |  train=720 val=128  |  lr=3e-05  |  test set skipped
Loading weights: 100% 100/100 [00:00<00:00, 17002.33it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
epoch 1: train_loss=3.0728 val_acc=0.2891 val_macro_f1=0.2215  <- best
epoch 2: train_loss=2

## Step 6 - Pick the winner by validation macro-F1

Only validation scores are used - neither run has seen the test set yet.
The winner's model goes to `models/bert/` and its validation results to `reports/`.
Both runs' validation metrics are also kept as `reports/bert_metrics_lr_<lr>.json`.

In [8]:
import json, shutil
from pathlib import Path

runs = {lr: json.loads(Path(f"runs/lr_{lr}/bert_metrics.json").read_text()) for lr in ["5e-5", "3e-5"]}

print(f"{'lr':<7}{'best epoch':>11}{'epochs run':>11}{'secs':>8}{'val acc':>9}{'val F1':>8}")
for lr, m in runs.items():
    print(f"{lr:<7}{m['best_epoch']:>11}{m['epochs_run']:>11}{m['train_seconds']:>8.0f}"
          f"{m['validation']['accuracy']:>9.4f}{m['validation']['macro_f1']:>8.4f}")

best_lr = max(runs, key=lambda lr: runs[lr]["validation"]["macro_f1"])  # ties -> 5e-5
print(f"Winner by validation macro-F1: lr={best_lr}")

shutil.rmtree("models/bert", ignore_errors=True)
shutil.copytree(f"runs/lr_{best_lr}/bert", "models/bert")
shutil.copy(f"runs/lr_{best_lr}/bert_metrics.json", "reports/bert_metrics.json")
shutil.copy(f"runs/lr_{best_lr}/bert_confusion_val.png", "reports/bert_confusion_val.png")
for lr in runs:
    shutil.copy(f"runs/lr_{lr}/bert_metrics.json", f"reports/bert_metrics_lr_{lr}.json")

print("Per-epoch history of the winner:")
for row in runs[best_lr]["history"]:
    print(row)

lr      best epoch epochs run    secs  val acc  val F1
5e-5             6          8      39   0.9688  0.9686
3e-5             7          8      41   0.9375  0.9367
Winner by validation macro-F1: lr=5e-5
Per-epoch history of the winner:
{'epoch': 1, 'train_loss': 3.0351, 'val_accuracy': 0.4062, 'val_macro_f1': 0.3382}
{'epoch': 2, 'train_loss': 2.1662, 'val_accuracy': 0.7734, 'val_macro_f1': 0.735}
{'epoch': 3, 'train_loss': 1.1861, 'val_accuracy': 0.8594, 'val_macro_f1': 0.8479}
{'epoch': 4, 'train_loss': 0.6089, 'val_accuracy': 0.9453, 'val_macro_f1': 0.9436}
{'epoch': 5, 'train_loss': 0.305, 'val_accuracy': 0.9531, 'val_macro_f1': 0.9536}
{'epoch': 6, 'train_loss': 0.1561, 'val_accuracy': 0.9688, 'val_macro_f1': 0.9686}
{'epoch': 7, 'train_loss': 0.0957, 'val_accuracy': 0.9531, 'val_macro_f1': 0.9536}
{'epoch': 8, 'train_loss': 0.0714, 'val_accuracy': 0.9609, 'val_macro_f1': 0.9611}


## Step 7 - Evaluate the winner ONCE on the test set

Loads `models/bert/` and adds the test results to `reports/bert_metrics.json`.
It refuses to run a second time on the same metrics file, so the test set is used once.

In [9]:
!python -m src.bert_model --test-only

Loading weights: 100% 104/104 [00:00<00:00, 4316.62it/s]
Validation  accuracy=0.9688  macro-F1=0.9686  (n=128)
Test        accuracy=0.9292  macro-F1=0.9277  (n=212)
Updated /content/ai-health-assistant/reports/bert_metrics.json
[
 {
  "true": "allergy",
  "predicted": "common cold",
  "count": 3
 },
 {
  "true": "chicken pox",
  "predicted": "dengue",
  "count": 1
 },
 {
  "true": "chicken pox",
  "predicted": "typhoid",
  "count": 1
 },
 {
  "true": "common cold",
  "predicted": "pneumonia",
  "count": 1
 },
 {
  "true": "dengue",
  "predicted": "cervical spondylosis",
  "count": 1
 }
]


In [10]:
m = json.loads(Path("reports/bert_metrics.json").read_text())
print(f"lr={m['params']['lr']}  best epoch={m['best_epoch']}  training time={m['train_seconds']} s")
for split in ("validation", "test"):
    print(f"{split:<11} accuracy={m[split]['accuracy']:.4f}  macro-F1={m[split]['macro_f1']:.4f}  (n={m[split]['n_samples']})")

lr=5e-05  best epoch=6  training time=38.8 s
validation  accuracy=0.9688  macro-F1=0.9686  (n=128)
test        accuracy=0.9292  macro-F1=0.9277  (n=212)


## Step 8 - Zip and download

Downloads `bert_results.zip` with `models/bert/` (about 250 MB) and `reports/`.
On your laptop, unzip it inside the project folder so the files land in `models/bert/` and `reports/`.

In [11]:
!zip -qr bert_results.zip models/bert reports
!ls -lh bert_results.zip
from google.colab import files
files.download("bert_results.zip")

-rw-r--r-- 1 root root 237M Nov  6 18:10 bert_results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>